# Aula 02: Árvores de Decisão & Reconhecimento de Letras

## 🎯 Objetivos de Aprendizagem
Ao final desta aula, você será capaz de:
- Explicar como uma Árvore de Decisão divide o espaço de features
- Treinar uma `DecisionTreeClassifier` e interpretar sua estrutura
- Visualizar e interpretar graficamente a árvore
- Analisar a importância das features para identificar letras
- Identificar overfitting variando a profundidade da árvore
- Comparar o comportamento de árvores rasas vs. profundas

## ✍️ Conexão com Escrita Manual
Na aula anterior, usamos k-NN para reconhecer dígitos — o modelo simplesmente mediu distâncias. Hoje usamos uma Árvore de
Decisão, que aprende **regras explícitas**: "se a proporção altura/largura é alta E há poucos pixels no quadrante
superior esquerdo → provavelmente é a letra 'i'". Essa interpretabilidade é muito valiosa em análise forense de
documentos, onde precisamos justificar por que um modelo tomou uma decisão.

---

# Parte 1: Setup & Motivação (15 min)

## De Dígitos para Letras

Na Aula 01 reconhecemos dígitos (0–9) com k-NN. Agora o desafio é maior: **26 classes** (letras A–Z), dataset de 20.000
amostras, e features estatísticas extraídas de imagens (não pixels brutos).

### O que muda?
| Aspecto | Aula 01 (k-NN) | Aula 02 (Decision Tree) |
|---|---|---|
| **Dataset** | sklearn `load_digits` — 1.797 amostras | UCI Letter Recognition — 20.000 amostras |
| **Features** | 64 pixels (8x8) | 16 medidas estatísticas de escrita |
| **Classes** | 10 dígitos | 26 letras |
| **Explicabilidade** | Nenhuma ("vizinhos mais próximos") | Alta (regras legíveis) |

### As 16 features do dataset Letter Recognition
Cada amostra é uma letra digitalizada descrita por 16 medidas:
- `x-box`, `y-box`, `width`, `height` — posição e tamanho da caixa delimitadora
- `onpix` — total de pixels ligados
- `x-bar`, `y-bar` — centróide horizontal e vertical
- `x2bar`, `y2bar` — variância horizontal e vertical
- `xybar` — correlação xy
- `x2ybr`, `xy2br` — momento de alta ordem
- `x-ege`, `xegvy`, `y-ege`, `yegvx` — contagens de bordas

> **Intuição:** Um perito grafista observa as mesmas características — largura do traço, proporção, distribuição dos
> pixels — para identificar letras. A árvore aprende automaticamente quais dessas medidas são mais discriminativas.

In [ ]:
# Environment verification — run this cell first
import importlib

required_packages = ["numpy", "pandas", "matplotlib", "seaborn", "sklearn", "ucimlrepo"]
missing = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]

if missing:
    raise ImportError(f"Missing packages: {missing}. Run: pip install {' '.join(missing)}")

print("Environment OK — all packages available")

In [ ]:
# Standard library
import warnings

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Dataset download
from ucimlrepo import fetch_ucirepo

# ML — core
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.tree import plot_tree

# ML — evaluation
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.dpi"] = 110

print("Imports OK")

## Carregando o Dataset Letter Recognition (UCI)

O `ucimlrepo` faz o download automaticamente na primeira execução e então usa cache local. Pode demorar alguns
segundos na primeira vez.

In [ ]:
# Fetch Letter Recognition dataset (UCI id=59)
letter_recognition = fetch_ucirepo(id=59)

X_raw: pd.DataFrame = letter_recognition.data.features
y_raw: pd.Series = letter_recognition.data.targets.squeeze()

print(f"Dataset shape: {X_raw.shape}")
print(f"Target shape:  {y_raw.shape}")
print(f"Classes ({y_raw.nunique()}): {sorted(y_raw.unique())}")
print(f"\nFeature names ({len(X_raw.columns)}):")
print(list(X_raw.columns))

In [ ]:
# Data exploration — always before any ML step
print("=== Class Distribution ===")
class_dist = y_raw.value_counts().sort_index()
print(class_dist)
print(f"\nMin samples per class: {class_dist.min()} ({class_dist.idxmin()})")
print(f"Max samples per class: {class_dist.max()} ({class_dist.idxmax()})")

print("\n=== Feature Statistics ===")
X_raw.describe()

### 📊 Plot 1: Distribuição das Classes

Antes de treinar qualquer modelo, checamos se as classes estão balanceadas. Desequilíbrio severo pode distorcer a
acurácia.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
class_dist.plot(kind="bar", ax=ax, edgecolor="white", linewidth=0.5)
ax.set_title("Distribuição das Classes — Letter Recognition Dataset", fontsize=13)
ax.set_xlabel("Letra", fontsize=11)
ax.set_ylabel("Número de Amostras", fontsize=11)
ax.axhline(class_dist.mean(), color="red", linestyle="--", linewidth=1.2, label=f"Média ({class_dist.mean():.0f})")
ax.legend()
plt.tight_layout()
plt.show()

**O que observar:** O dataset é aproximadamente balanceado (~769 amostras por classe). Isso é bom — significa que a
acurácia simples é uma métrica confiável. Se houvesse classes muito raras, precisaríamos de métricas como F1-score ou
métricas balanceadas.

---
# Parte 2: Implementação Prática (55 min)

## Como Funciona uma Árvore de Decisão?

A árvore aprende a fazer perguntas binárias sobre as features para separar as classes. Em cada nó, ela escolhe a
feature e o limiar que **maximiza a pureza** dos grupos resultantes (medida pelo índice Gini ou Entropia).

```
                 [onpix <= 3.5?]
                /               \
           SIM (Gini=0.42)    NÃO (Gini=0.51)
              /                     \
      [x-ege <= 2.0?]         [height <= 6.5?]
         /     \                  /       \
       'i'     'l'              'O'       'M'
```

**Índice Gini:** mede a impureza de um nó. Gini = 0 → nó puro (todas as amostras são da mesma classe). Gini = 0.5 → 
máxima impureza (classes uniformemente distribuídas).

In [ ]:
# Convert to numpy for sklearn
X: np.ndarray = X_raw.values
y: np.ndarray = y_raw.values

# Train/test split — stratified to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,  # Ensures each class has proportional representation in both sets
)

print(f"Training set:  {X_train.shape[0]} samples")
print(f"Test set:      {X_test.shape[0]} samples")
print(f"Features:      {X_train.shape[1]}")
print(f"Classes:       {len(np.unique(y_train))}")

## Baseline: Árvore sem Restrição de Profundidade

Começamos simples: treinamos uma árvore que pode crescer livremente (`max_depth=None`). Ela vai memorizar o dado de
treino — é nosso ponto de partida para entender overfitting.

In [ ]:
def train_decision_tree(
    X_train: np.ndarray,
    y_train: np.ndarray,
    max_depth: int | None = None,
    criterion: str = "gini",
    random_state: int = 42,
) -> DecisionTreeClassifier:
    """Train a DecisionTreeClassifier and return the fitted model."""
    model = DecisionTreeClassifier(
        max_depth=max_depth,
        criterion=criterion,
        random_state=random_state,
    )
    model.fit(X_train, y_train)
    return model


# Baseline — unrestricted depth
tree_full = train_decision_tree(X_train, y_train, max_depth=None)

train_acc_full = accuracy_score(y_train, tree_full.predict(X_train))
test_acc_full = accuracy_score(y_test, tree_full.predict(X_test))

print("=== Baseline: Unrestricted Depth ===")
print(f"Tree depth:         {tree_full.get_depth()}")
print(f"Number of leaves:   {tree_full.get_n_leaves()}")
print(f"Train accuracy:     {train_acc_full:.3f}")
print(f"Test accuracy:      {test_acc_full:.3f}")
print()
print("⚠️  Train= 1.000, Test < 1.000 → classic overfitting signature!")

## Controlando a Profundidade: Curva Underfitting → Overfitting

O parâmetro `max_depth` é o principal controle de complexidade da árvore. Vamos medir a acurácia de treino e teste
para várias profundidades e olhar para o **ponto ótimo** onde o gap entre treino e teste é mínimo.

In [ ]:
depths = list(range(1, 31))
train_scores: list[float] = []
test_scores: list[float] = []

for depth in depths:
    model = train_decision_tree(X_train, y_train, max_depth=depth)
    train_scores.append(accuracy_score(y_train, model.predict(X_train)))
    test_scores.append(accuracy_score(y_test, model.predict(X_test)))

best_depth = depths[int(np.argmax(test_scores))]
best_test_acc = max(test_scores)
print(f"Best test accuracy: {best_test_acc:.3f} at max_depth={best_depth}")

### 📊 Plot 2: Curva de Acurácia Treino vs. Teste por Profundidade

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(depths, train_scores, marker="o", markersize=4, label="Train accuracy", linewidth=2)
ax.plot(depths, test_scores, marker="s", markersize=4, label="Test accuracy", linewidth=2)
ax.axvline(best_depth, color="gray", linestyle="--", linewidth=1.2, label=f"Best depth = {best_depth}")
ax.set_title("Curva de Acurácia Treino vs. Teste — Decision Tree (Letter Recognition)", fontsize=13)
ax.set_xlabel("max_depth", fontsize=11)
ax.set_ylabel("Acurácia", fontsize=11)
ax.set_ylim(0.0, 1.05)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

**O que observar:**
- **Profundidade baixa (1–4):** alto bias — a árvore é simples demais, erra muito treino E teste (**underfitting**).
- **Profundidade ideal (~10–15):** a acurácia de teste alcança seu pico. A árvore aprendeu padrões generalizáveis.
- **Profundidade alta (> 20):** acurácia de treino → 100%, mas acurácia de teste cai. A árvore memorizou o treino
  (**overfitting**).

> **Bias-variance tradeoff:** aumentar `max_depth` reduz o bias (modelo mais expressivo) mas aumenta a variância
> (mais sensível ao ruído do treino). O ponto ideal equilibra os dois.

## Modelo Final: Melhor Profundidade

In [ ]:
# Train the best model
tree_best = train_decision_tree(X_train, y_train, max_depth=best_depth)

y_pred: np.ndarray = tree_best.predict(X_test)

print(f"=== Best Model: max_depth={best_depth} ===")
print(f"Train accuracy: {accuracy_score(y_train, tree_best.predict(X_train)):.3f}")
print(f"Test accuracy:  {accuracy_score(y_test, y_pred):.3f}")
print()
# Macro average — useful for multi-class, treats all classes equally
print(classification_report(y_test, y_pred))

## Visualizando a Árvore

Uma das maiores vantagens das Árvores de Decisão é a **interpretabilidade**: podemos ler as regras aprendidas. Vamos
visualizar os primeiros 3 níveis da árvore para entender quais features guiam as primeiras decisões.

In [ ]:
feature_names: list[str] = list(X_raw.columns)
class_names: list[str] = sorted(np.unique(y_train))

fig, ax = plt.subplots(figsize=(28, 10))
plot_tree(
    tree_best,
    feature_names=feature_names,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=8,
    max_depth=3,  # Show only first 3 levels for readability
    ax=ax,
)
ax.set_title(
    f"Árvore de Decisão (max_depth={best_depth}) — Primeiros 3 Níveis",
    fontsize=14,
)
plt.tight_layout()
plt.show()

**Como ler a visualização:**
- Cada nó mostra: **feature usada para dividir**, **limiar**, **índice Gini** (impureza), **amostras** e **classe
  majoritária**.
- Nós coloridos: a intensidade da cor representa a pureza — quanto mais saturado, mais puro o nó.
- As **primeiras divisões** são as mais informativas — a árvore começa pelas features que mais separam as classes.

## Importância das Features

`feature_importances_` mede a contribuição de cada feature para reduzir a impureza (Gini) ao longo de toda a árvore.
Features com alta importância são as que a árvore mais usa para discriminar entre letras — equivale a identificar quais
características da escrita são mais distintivas.

In [ ]:
importances: np.ndarray = tree_best.feature_importances_
fi_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

print("=== Feature Importances ===")
print(fi_df.to_string(index=False))

### 📊 Plot 3: Importância das Features

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(
    data=fi_df,
    x="importance",
    y="feature",
    ax=ax,
    orient="h",
)
ax.set_title(f"Importância das Features — Decision Tree (max_depth={best_depth})", fontsize=13)
ax.set_xlabel("Importância (redução média de Gini)", fontsize=11)
ax.set_ylabel("Feature", fontsize=11)
plt.tight_layout()
plt.show()

**O que observar:**
- As features mais importantes são tipicamente relacionadas ao **centróide** (`x-bar`, `y-bar`) e à **distribuição de
  pixels** — exatamente o que um perito grafista usaria para distinguir letras.
- Features com importância próxima de zero são redundantes para este modelo — podem ser removidas sem perda
  significativa de acurácia.

> **Conexão forense:** Em análise de documentos, identificar as features mais discriminativas permite que peritos
> construam argumentos rastreáveis — "identificamos a letra X pois a medida Y do suspeito apresenta valor Z".

## Análise de Erros: Confusion Matrix

A matriz de confusão mostra **para cada classe real, como o modelo classificou as amostras**. Com 26 classes, a
visualização é densa — vamos focar nos pares de classes mais confundidos.

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=class_names)

fig, ax = plt.subplots(figsize=(16, 13))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(
    ax=ax,
    colorbar=True,
    cmap="Blues",
    xticks_rotation=45,
)
ax.set_title(f"Confusion Matrix — Decision Tree (max_depth={best_depth})", fontsize=13)
plt.tight_layout()
plt.show()

**Como interpretar:**
- Diagonal principal: acertos (linha = classe real, coluna = classe predita).
- Células fora da diagonal: erros — `cm[i][j]` = amostras da classe `i` classificadas como `j`.
- Pares mais confundidos costumam ser letras visualmente similares: **D/O**, **I/J**, **C/G**, **U/V**.

Vamos identificar programaticamente os maiores erros:

In [ ]:
# Find worst confusion pairs (excluding diagonal)
cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
np.fill_diagonal(cm_df.values, 0)  # Zero out the diagonal (correct predictions)

# Stack and sort by confusion count
pairs = (
    cm_df.stack()
    .reset_index()
    .rename(columns={"level_0": "true_label", "level_1": "predicted_label", 0: "count"})
    .query("count > 0")
    .sort_values("count", ascending=False)
    .head(10)
)

print("=== Top 10 Most Confused Letter Pairs ===")
print(pairs.to_string(index=False))

---
## ✏️ Try This 1: Critério de Impureza — Gini vs. Entropy

O `criterion` controla como a árvore mede a qualidade de cada divisão:
- `'gini'` (padrão): índice Gini — computacionalmente mais rápido
- `'entropy'`: ganho de informação — em teoria mais rigoroso

Na prática, a diferença de acurácia costuma ser pequena. Teste e observe:

In [ ]:
# ✏️ Try This: Change criterion from 'gini' to 'entropy' and compare accuracy

for criterion in ["gini", "entropy"]:
    model = train_decision_tree(X_train, y_train, max_depth=best_depth, criterion=criterion)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"criterion='{criterion}': test accuracy = {test_acc:.4f}")

# What do you observe? Which criterion performs better for this dataset?
# Does the difference surprise you?

---
## ✏️ Try This 2: Efeito do min_samples_split

`min_samples_split` define o número mínimo de amostras que um nó precisa ter para ser dividido. Valores maiores forçam
a árvore a fazer divisões mais conservadoras, reduzindo overfitting.

In [ ]:
# ✏️ Try This: Vary min_samples_split and observe the trade-off

for min_split in [2, 5, 10, 50, 200]:
    model = DecisionTreeClassifier(
        max_depth=best_depth,
        min_samples_split=min_split,
        random_state=42,
    )
    model.fit(X_train, y_train)
    train_acc = accuracy_score(y_train, model.predict(X_train))
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"min_samples_split={min_split:>4}: train={train_acc:.3f}  test={test_acc:.3f}  leaves={model.get_n_leaves()}")

# What happens to the number of leaves as min_samples_split increases?
# Can you increase test accuracy by tuning this parameter?

---
## ✏️ Try This 3: Top Features Apenas

Se as features mais importantes já carregam a maior parte da informação, a árvore deve funcionar bem usando apenas elas.
Vamos testar!

In [ ]:
# ✏️ Try This: Use only the top N most important features

top_n_options = [3, 5, 8, 16]  # Try changing these values

for top_n in top_n_options:
    top_features: list[str] = fi_df["feature"].head(top_n).tolist()
    top_indices = [feature_names.index(f) for f in top_features]

    model = train_decision_tree(X_train[:, top_indices], y_train, max_depth=best_depth)
    test_acc = accuracy_score(y_test, model.predict(X_test[:, top_indices]))
    print(f"Top {top_n:>2} features: test accuracy = {test_acc:.3f}")

# Does removing low-importance features hurt accuracy much?
# What does this suggest about feature redundancy in this dataset?

---
# Parte 3: Interpretação & Discussão (15 min)

## Análise dos Resultados

In [ ]:
# Per-class accuracy — which letters are hardest to recognize?
report_dict = classification_report(y_test, y_pred, output_dict=True)
per_class_df = (
    pd.DataFrame(report_dict)
    .T
    .drop(index=["accuracy", "macro avg", "weighted avg"])
    .sort_values("f1-score", ascending=True)
    .head(10)
)

print("=== 10 Hardest Letters (lowest F1-score) ===")
print(per_class_df[["precision", "recall", "f1-score", "support"]].to_string())

## 🔬 Padrões de Erro e Conexão com Escrita Manual

**Por que certas letras são mais difíceis?**

As features do dataset capturam medidas geométricas simples (centróide, proporções, bordas). Letras com formas
geometricamente similares a outras são inerentemente mais difíceis de separar com divisões lineares por feature:

- **D/O:** ambas têm forma arredondada, centróides similares
- **I/J:** alturas similares, poucas bordas horizontais
- **C/G:** curvas quase idênticas
- **U/V**: pixels distribuídos de forma similar no eixo vertical

**Conexão forense:** Em análise de documentos suspeitos, erros nesses pares problemáticos precisam de confirmação
humana. O modelo pode servir como triagem inicial — elimina candidatos óbvios mas não substitui o perito para casos
ambíguos.

## 🚫 Equívocos Comuns dos Alunos

1. **"Acurácia de treino 100% significa modelo perfeito"** — Na verdade significa overfitting: o modelo memorizou o
   treino mas não generalizou.

2. **"Depth maior é sempre melhor"** — Falso. Profundidade excessiva aumenta a variância sem ganho real no teste.

3. **"Feature importance indica causalidade"** — Apenas indica correlação útil para as divisões da árvore. Uma feature
   pode ser importante por razões espúrias no dataset.

## 🤔 Think About It

Estas perguntas não exigem código — são para reflexão e discussão em aula:

1. **Interpretabilidade vs. Acurácia:** Uma árvore com `max_depth=5` é menos precisa que uma com `max_depth=20`, mas
   muito mais legível. Em que cenários essa troca de acurácia por interpretabilidade seria obrigatória? (Pense em
   sistemas judiciais, médicos, financeiros.)

2. **Feature engineering:** As 16 features do dataset foram criadas manualmente por pesquisadores. O que aconteceria
   se pudéssemos usar os pixels brutos (como na Aula 01)? Qual seria a desvantagem?

3. **Generalização:** O modelo foi treinado e testado com amostras da mesma distribuição. Se aplicarmos o modelo a
   letras escritas à mão por pessoas de outros países (alfabeto diferente, caligrafia distinta), o que esperamos?

## 🎯 Challenge (Opcional)

Treine **duas** árvores: uma com `max_depth=3` (interpretável) e outra com a melhor profundidade encontrada. Para cada
árvore:
1. Extraia as regras dos primeiros 2 níveis (`tree_.threshold`, `tree_.feature`)
2. Traduza as regras para linguagem natural: "Se `x-bar` ≤ 7.5, então..."
3. Interprete: o que essas regras dizem sobre como a árvore "enxerga" as letras?

---
# Parte 4: Encerramento & Preview (5 min)

## ✅ Conceitos-Chave desta Aula

- **Árvore de Decisão** aprende regras binárias explícitas — ao contrário do k-NN, explica *por que* classificou assim
- **`max_depth`** é o principal controle de complexidade: baixo → underfitting, alto → overfitting
- **Bias-variance tradeoff**: diminuir o bias aumenta a variância — encontrar o equilíbrio é o objetivo
- **Feature importance** identifica as medidas de escrita mais discriminativas
- **Confusion matrix** revela quais classes são confundidas — pares problemáticos geralmente são visualmente similares

## 📝 Tarefa

Escolha uma das opções:

**Opção A — Experimento:** Treine árvores com `max_depth` em `[2, 5, 10, 20]` **e** `criterion` em
`['gini', 'entropy']` (grade 4×2). Plote uma heatmap de acurácia de teste para cada combinação. Qual par de
hiperparâmetros maximiza o desempenho no teste?

**Opção B — Reflexão:** Em análise forense de documentos, um perito precisa justificar sua conclusão em tribunal. Como
a estrutura de regras de uma Árvore de Decisão poderia ser usada para gerar um relatório de evidências automático? Que
limitações esse sistema teria?

## 🔭 Preview: Aula 03 — Regressão Linear & Análise de Pressão de Caneta

Na próxima aula saímos da classificação e entramos na **regressão**: em vez de prever uma categoria, vamos prever
**valores contínuos** — como a pressão exercida ao escrever, que pode revelar traços de personalidade e estado
emocional do escritor.